# 01 — Auditoria e preparação dos dados

Esta etapa valida estrutura, tipos, granularidade, snapshots, chaves e qualidade dos cinco CSVs. Os arquivos de `data/` são apenas lidos. Todas as saídas derivadas são gravadas em `outputs/auditoria/`.

Limite desta etapa: ainda não há recomendação de investimento, cálculo de retorno ou escolha de segmento.

In [1]:
from pathlib import Path
import hashlib
import unicodedata

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_colwidth', 120)


def encontrar_raiz(inicio=None):
    caminho = Path(inicio or Path.cwd()).resolve()
    for candidato in (caminho, *caminho.parents):
        if (candidato / 'data').is_dir() and (candidato / 'ROADMAP.md').is_file():
            return candidato
    raise FileNotFoundError('Raiz do projeto não encontrada.')


def sha256(caminho, tamanho_bloco=1024 * 1024):
    resumo = hashlib.sha256()
    with caminho.open('rb') as arquivo:
        for bloco in iter(lambda: arquivo.read(tamanho_bloco), b''):
            resumo.update(bloco)
    return resumo.hexdigest().upper()


RAIZ = encontrar_raiz()
DADOS = RAIZ / 'data'
SAIDAS = RAIZ / 'outputs' / 'auditoria'
SAIDAS.mkdir(parents=True, exist_ok=True)

ARQUIVOS = {
    'details': 'Details_Itapema.csv',
    'hosts': 'Hosts_ids_Itapema.csv',
    'mesh': 'Mesh_Ids_Data_Itapema.csv',
    'precos': 'Price_AV_Itapema.csv',
    'vendas': 'VivaReal_Itapema.csv',
}

HASHES_ETAPA_0 = {
    'Details_Itapema.csv': '7A28A35811B5B01CA046D06E0AF80180E43D07AF6923FC03B76DF99AC01050C9',
    'Hosts_ids_Itapema.csv': 'B2E5AA3E0BD30A3FA63643ABC4BC3142C78BE165855BBD6C4D077D6BDE308EA9',
    'Mesh_Ids_Data_Itapema.csv': '7C9DAA0D37FE5C8FA10E6EFA53CB9E6F66E28880E165A62D3E1F9C74585ADF1E',
    'Price_AV_Itapema.csv': 'B0B5C8C07011DAF5C91F2FB9E7BA735026F0AE4542745481376140A714DD813B',
    'VivaReal_Itapema.csv': 'C720320AE6BCD34982323A2D6EEC6D5F5F18E316B3A3DAE0A37F03638E32A631',
}

caminhos = {nome: DADOS / arquivo for nome, arquivo in ARQUIVOS.items()}
ausentes = [str(caminho) for caminho in caminhos.values() if not caminho.is_file()]
assert not ausentes, f'Arquivos ausentes: {ausentes}'
hashes_antes = {caminho.name: sha256(caminho) for caminho in caminhos.values()}
assert hashes_antes == HASHES_ETAPA_0, 'Os dados brutos diferem dos hashes validados na Etapa 0.'

print(f'Raiz: {RAIZ}')
print(f'Saídas permitidas: {SAIDAS}')
print('Hashes iniciais confirmados.')

Raiz: C:\Users\rewel\Documents\jt2026-Antonio-Rewelli-Santos
Saídas permitidas: C:\Users\rewel\Documents\jt2026-Antonio-Rewelli-Santos\outputs\auditoria
Hashes iniciais confirmados.


In [2]:
CONFIG = {
    'details': {
        'ids': ['airbnb_listing_id', 'owner_id'],
        'datas': ['aquisition_date'],
        'booleanos': ['can_instant_book', 'is_professional', 'is_new_listing', 'is_guest_favorite'],
        'numericos': [
            'number_of_bathrooms', 'number_of_bedrooms', 'number_of_beds', 'latitude', 'longitude',
            'number_of_guests', 'number_of_reviews', 'cleaning_fee', 'star_rating', 'picture_count',
            'min_nights', 'guest_satisfaction_overall', 'accuracy_rating', 'checkin_rating',
            'cleanliness_rating', 'communication_rating', 'location_rating', 'value_rating',
        ],
    },
    'hosts': {
        'ids': ['owner_id'],
        'datas': ['host_snapshot_date'],
        'booleanos': ['is_superhost', 'is_verified'],
        'numericos': ['number_of_reviews_host', 'star_rating_host', 'years_host', 'months_host'],
        'percentuais': ['response_rate_shown'],
    },
    'mesh': {
        'ids': ['airbnb_listing_id'],
        'datas': ['aquisition_date'],
        'booleanos': [],
        'numericos': ['latitude', 'longitude'],
    },
    'precos': {
        'ids': ['airbnb_listing_id'],
        'datas': ['date', 'aquisition_date'],
        'booleanos': [],
        'numericos': ['price'],
    },
    'vendas': {
        'ids': ['listing_id'],
        'datas': ['aquisition_date'],
        'booleanos': [],
        'numericos': [
            'sale_price', 'rental_price', 'yearly_iptu', 'monthly_condo_fee', 'usable_area',
            'bathrooms', 'bedrooms', 'parking_spaces',
        ],
    },
}


def converter_booleano(serie):
    mapa = {'true': True, 'false': False, '1': True, '0': False, 'sim': True, 'não': False, 'nao': False}
    normalizada = serie.astype('string').str.strip().str.lower()
    return normalizada.map(mapa).astype('boolean')


def converter_percentual(serie):
    texto = serie.astype('string').str.strip().str.replace('%', '', regex=False).str.replace(',', '.', regex=False)
    return pd.to_numeric(texto, errors='coerce')


def normalizar_chave_texto(serie):
    marcadores_ausentes = {'', 'none', 'null', 'nan', '<na>'}

    def normalizar(valor):
        if pd.isna(valor):
            return pd.NA
        texto = str(valor).strip()
        if texto.casefold() in marcadores_ausentes:
            return pd.NA
        sem_acentos = ''.join(
            caractere for caractere in unicodedata.normalize('NFKD', texto)
            if not unicodedata.combining(caractere)
        )
        chave = ''.join(caractere for caractere in sem_acentos.casefold() if caractere.isalnum())
        return chave or pd.NA

    return serie.map(normalizar).astype('string')


def carregar_e_tipar(nome):
    config = CONFIG[nome]
    tipos_ids = {coluna: 'string' for coluna in config['ids']}
    bruto = pd.read_csv(
        caminhos[nome],
        dtype=tipos_ids,
        keep_default_na=True,
        na_values=['<NA>', ''],
        low_memory=False,
    )
    tratado = bruto.copy(deep=True)

    for coluna in config['ids']:
        tratado[coluna] = tratado[coluna].astype('string').str.strip().replace('', pd.NA)
    for coluna in config['datas']:
        tratado[coluna] = pd.to_datetime(tratado[coluna], errors='coerce', format='mixed')
    for coluna in config['numericos']:
        tratado[coluna] = pd.to_numeric(tratado[coluna], errors='coerce')
    for coluna in config['booleanos']:
        tratado[coluna] = converter_booleano(tratado[coluna])
    for coluna in config.get('percentuais', []):
        tratado[coluna] = converter_percentual(tratado[coluna])

    colunas_tipadas = set(
        config['ids'] + config['datas'] + config['numericos'] + config['booleanos'] + config.get('percentuais', [])
    )
    for coluna in tratado.columns.difference(list(colunas_tipadas)):
        if tratado[coluna].dtype == 'object' or isinstance(tratado[coluna].dtype, pd.StringDtype):
            tratado[coluna] = (
                tratado[coluna].astype('string').str.replace(r'\s+', ' ', regex=True).str.strip().replace('', pd.NA)
            )
    if 'suburb' in tratado.columns:
        tratado['suburb_key'] = normalizar_chave_texto(tratado['suburb'])
    return bruto, tratado


brutos = {}
tratados = {}
for nome in ARQUIVOS:
    brutos[nome], tratados[nome] = carregar_e_tipar(nome)

resumo_carga = pd.DataFrame(
    [
        {
            'dataset': nome,
            'arquivo': ARQUIVOS[nome],
            'linhas': len(tratados[nome]),
            'colunas': tratados[nome].shape[1],
            'memoria_mb': tratados[nome].memory_usage(deep=True).sum() / 1024**2,
        }
        for nome in ARQUIVOS
    ]
)
display(resumo_carga)

,dataset,arquivo,linhas,colunas,memoria_mb
0,details,Details_Itapema.csv,4441,35,10.229647
1,hosts,Hosts_ids_Itapema.csv,4440,11,0.750545
2,mesh,Mesh_Ids_Data_Itapema.csv,4441,9,1.605035
3,precos,Price_AV_Itapema.csv,118839,4,9.785109
4,vendas,VivaReal_Itapema.csv,8329,23,9.492824


In [3]:
registros_qualidade = []


def registrar(dataset, metrica, valor, coluna='', severidade='informativo', detalhe=''):
    registros_qualidade.append(
        {
            'dataset': dataset,
            'coluna': coluna,
            'metrica': metrica,
            'valor': valor,
            'severidade': severidade,
            'detalhe': detalhe,
        }
    )


def contar_mascara(mascara):
    return int(mascara.fillna(False).sum())


def papel_unidade_regra(nome, coluna):
    config = CONFIG[nome]
    if coluna == 'suburb_key':
        return 'chave derivada', 'texto normalizado', 'remoção de acentos, espaços e pontuação; original preservado'
    if coluna in config['ids']:
        return 'identificador', 'sem unidade', 'texto sem espaços laterais; ausentes preservados'
    if coluna in config['datas']:
        return 'data', 'data/hora', 'datetime; valores não interpretáveis viram ausentes'
    if coluna in config['booleanos']:
        return 'indicador', 'booleano', 'normalização explícita de verdadeiro/falso'
    if coluna in config.get('percentuais', []):
        return 'métrica', 'percentual', 'remoção do símbolo % e conversão numérica'
    if coluna in config['numericos']:
        if coluna in {'price', 'sale_price', 'rental_price', 'yearly_iptu', 'monthly_condo_fee', 'cleaning_fee'}:
            unidade = 'BRL anunciado'
        elif coluna in {'latitude', 'longitude'}:
            unidade = 'graus decimais'
        elif coluna == 'usable_area':
            unidade = 'm²'
        elif 'rating' in coluna or coluna == 'guest_satisfaction_overall':
            unidade = 'escala de avaliação'
        else:
            unidade = 'contagem/valor numérico'
        return 'métrica', unidade, 'conversão numérica; inválidos viram ausentes'
    return 'atributo', 'texto/categoria', 'texto sem espaços laterais; ausentes preservados'


dicionario = []
for nome, df in tratados.items():
    bruto = brutos[nome]
    registrar(nome, 'linhas', len(df))
    registrar(nome, 'colunas', df.shape[1])
    registrar(nome, 'duplicatas_exatas', int(bruto.duplicated().sum()), severidade='atenção')

    for coluna in df.columns:
        nulos = int(df[coluna].isna().sum())
        registrar(nome, 'nulos', nulos, coluna, 'atenção' if nulos else 'informativo')
        registrar(nome, 'nulos_percentual', round(nulos / len(df) * 100, 4) if len(df) else 0, coluna)
        registrar(nome, 'valores_unicos', int(df[coluna].nunique(dropna=True)), coluna)
        papel, unidade, regra = papel_unidade_regra(nome, coluna)
        dicionario.append(
            {
                'dataset': nome,
                'coluna': coluna,
                'descricao': coluna.replace('_', ' '),
                'papel': papel,
                'unidade': unidade,
                'dtype_original': str(bruto[coluna].dtype) if coluna in bruto.columns else 'derivada',
                'dtype_tratado': str(df[coluna].dtype),
                'regra_tratamento': regra,
            }
        )

    config = CONFIG[nome]
    for coluna in config['datas'] + config['numericos'] + config['booleanos'] + config.get('percentuais', []):
        invalidos = bruto[coluna].notna() & df[coluna].isna()
        registrar(nome, 'valores_nao_interpretaveis', contar_mascara(invalidos), coluna, 'atenção')
    for coluna in config['datas']:
        registrar(nome, 'data_minima', df[coluna].min(), coluna)
        registrar(nome, 'data_maxima', df[coluna].max(), coluna)

# Regras de domínio: os registros são sinalizados, não removidos silenciosamente.
for nome, coluna in [
    ('precos', 'price'), ('vendas', 'sale_price'), ('vendas', 'rental_price'),
    ('vendas', 'yearly_iptu'), ('vendas', 'monthly_condo_fee'), ('details', 'cleaning_fee'),
]:
    serie = tratados[nome][coluna]
    registrar(nome, 'valores_negativos', contar_mascara(serie < 0), coluna, 'erro')

for nome, coluna in [
    ('details', 'number_of_bathrooms'), ('details', 'number_of_bedrooms'), ('details', 'number_of_beds'),
    ('details', 'number_of_guests'), ('details', 'min_nights'), ('vendas', 'usable_area'),
    ('vendas', 'bathrooms'), ('vendas', 'bedrooms'), ('vendas', 'parking_spaces'),
]:
    registrar(nome, 'valores_negativos', contar_mascara(tratados[nome][coluna] < 0), coluna, 'erro')

for nome in ['details', 'mesh']:
    registrar(nome, 'latitude_fora_limite', contar_mascara(~tratados[nome]['latitude'].between(-90, 90)), 'latitude', 'erro')
    registrar(nome, 'longitude_fora_limite', contar_mascara(~tratados[nome]['longitude'].between(-180, 180)), 'longitude', 'erro')
    coordenadas_zeradas = tratados[nome]['latitude'].eq(0) & tratados[nome]['longitude'].eq(0)
    registrar(nome, 'pares_coordenadas_zeradas', contar_mascara(coordenadas_zeradas), 'latitude + longitude', 'erro')
    registrar(nome, 'latitude_minima', tratados[nome]['latitude'].min(), 'latitude')
    registrar(nome, 'latitude_maxima', tratados[nome]['latitude'].max(), 'latitude')
    registrar(nome, 'longitude_minima', tratados[nome]['longitude'].min(), 'longitude')
    registrar(nome, 'longitude_maxima', tratados[nome]['longitude'].max(), 'longitude')

for nome in ['mesh', 'vendas']:
    df = tratados[nome]
    bairros_validos = df.loc[df['suburb_key'].notna(), ['suburb', 'suburb_key']].drop_duplicates()
    variantes = bairros_validos.groupby('suburb_key')['suburb'].nunique()
    chaves_com_variantes = variantes[variantes > 1].index
    exemplos = (
        bairros_validos[bairros_validos['suburb_key'].isin(chaves_com_variantes)]
        .groupby('suburb_key')['suburb']
        .agg(lambda valores: ' | '.join(sorted(set(valores.astype(str)))))
        .head(10)
        .to_dict()
    )
    registrar(nome, 'bairros_originais_distintos', int(df['suburb'].nunique(dropna=True)), 'suburb')
    registrar(nome, 'bairros_chave_distintos', int(df['suburb_key'].nunique(dropna=True)), 'suburb_key')
    registrar(nome, 'chaves_bairro_com_variantes', len(chaves_com_variantes), 'suburb_key', 'atenção', str(exemplos))
    cidade_key = normalizar_chave_texto(df['city'])
    registrar(nome, 'cidade_diferente_de_itapema', contar_mascara(cidade_key.notna() & (cidade_key != 'itapema')), 'city', 'atenção')

bairros_fontes = pd.concat(
    [
        tratados['mesh'][['suburb', 'suburb_key']].assign(fonte='mesh'),
        tratados['vendas'][['suburb', 'suburb_key']].assign(fonte='vendas'),
    ],
    ignore_index=True,
).dropna(subset=['suburb_key'])
rotulos_por_chave = bairros_fontes.drop_duplicates().groupby('suburb_key')['suburb'].agg(
    lambda valores: sorted(set(valores.astype(str)))
)
variantes_entre_fontes = rotulos_por_chave[rotulos_por_chave.map(len) > 1]
registrar(
    'bairros',
    'chaves_com_rotulos_variantes_entre_fontes',
    len(variantes_entre_fontes),
    'suburb_key',
    'atenção',
    str(variantes_entre_fontes.head(15).to_dict()),
)

ratings_0_5 = [
    ('details', 'star_rating'), ('details', 'accuracy_rating'), ('details', 'checkin_rating'),
    ('details', 'cleanliness_rating'), ('details', 'communication_rating'),
    ('details', 'location_rating'), ('details', 'value_rating'), ('hosts', 'star_rating_host'),
]
for nome, coluna in ratings_0_5:
    serie = tratados[nome][coluna]
    registrar(nome, 'fora_escala_0_5', contar_mascara(serie.notna() & ~serie.between(0, 5)), coluna, 'erro')
    registrar(nome, 'valor_zero_possivel_sem_avaliacao', contar_mascara(serie == 0), coluna, 'atenção')

satisfacao = tratados['details']['guest_satisfaction_overall']
registrar('details', 'fora_escala_0_100', contar_mascara(satisfacao.notna() & ~satisfacao.between(0, 100)), 'guest_satisfaction_overall', 'erro')
captura_apos_estadia = tratados['precos']['aquisition_date'] > tratados['precos']['date']
registrar('precos', 'captura_apos_data_estadia', contar_mascara(captura_apos_estadia), 'aquisition_date', 'atenção')

for nome, coluna in [
    ('precos', 'price'), ('vendas', 'sale_price'), ('vendas', 'usable_area'),
    ('details', 'cleaning_fee'), ('details', 'number_of_bedrooms'),
]:
    serie = tratados[nome][coluna].dropna()
    if len(serie):
        q1, q3 = serie.quantile([0.25, 0.75])
        iqr = q3 - q1
        limite_inferior, limite_superior = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        quantidade = int(((serie < limite_inferior) | (serie > limite_superior)).sum())
        registrar(nome, 'outliers_iqr', quantidade, coluna, 'atenção', f'limites: {limite_inferior:.4f} a {limite_superior:.4f}')

qualidade_preliminar = pd.DataFrame(registros_qualidade)
display(qualidade_preliminar.groupby(['dataset', 'severidade'], dropna=False).size().unstack(fill_value=0))

severidade,atenção,erro,informativo
dataset,,,
bairros,1,0,0
details,40,17,106
hosts,12,1,35
mesh,7,3,36
precos,6,1,18
vendas,21,8,68


In [4]:
def registrar_chave(nome, df, chaves):
    rotulo = ' + '.join(chaves)
    faltantes = df[chaves].isna().any(axis=1)
    duplicadas = df.loc[~faltantes].duplicated(chaves, keep=False)
    registrar(nome, 'chave_ausente', int(faltantes.sum()), rotulo, 'erro')
    registrar(nome, 'linhas_em_chaves_duplicadas', int(duplicadas.sum()), rotulo, 'atenção')


def snapshot_mais_recente(nome, chave, coluna_data):
    df = tratados[nome].copy()
    registrar_chave(nome, df, [chave])
    validos = df.loc[df[chave].notna()].copy()
    contagens = validos.groupby(chave, dropna=False).size().rename('quantidade_snapshots').reset_index()
    mais_recente = (
        validos.sort_values([chave, coluna_data], kind='stable', na_position='first')
        .drop_duplicates(chave, keep='last')
        .merge(contagens, on=chave, how='left', validate='one_to_one')
    )
    registrar(nome, 'linhas_apos_snapshot', len(mais_recente))
    registrar(nome, 'entidades_com_multiplos_snapshots', int((contagens['quantidade_snapshots'] > 1).sum()), chave, 'atenção')
    assert mais_recente[chave].is_unique
    return mais_recente


details = snapshot_mais_recente('details', 'airbnb_listing_id', 'aquisition_date')
hosts = snapshot_mais_recente('hosts', 'owner_id', 'host_snapshot_date')
mesh = snapshot_mais_recente('mesh', 'airbnb_listing_id', 'aquisition_date')
vendas = snapshot_mais_recente('vendas', 'listing_id', 'aquisition_date')

precos_raw = tratados['precos'].copy()
chave_preco = ['airbnb_listing_id', 'date']
registrar_chave('precos', precos_raw, chave_preco)
precos_validos = precos_raw.loc[precos_raw[chave_preco].notna().all(axis=1)].copy()
estatisticas_captura = (
    precos_validos.groupby(chave_preco, dropna=False)['aquisition_date']
    .agg(quantidade_capturas='size', primeira_captura='min', ultima_captura='max')
    .reset_index()
)
precos = (
    precos_validos.sort_values(chave_preco + ['aquisition_date'], kind='stable', na_position='first')
    .drop_duplicates(chave_preco, keep='last')
    .merge(estatisticas_captura, on=chave_preco, how='left', validate='one_to_one')
)
precos['flag_preco_invalido'] = precos['price'].isna() | (precos['price'] <= 0)
precos['flag_captura_apos_estadia'] = precos['aquisition_date'] > precos['date']
vendas['flag_preco_venda_invalido'] = vendas['sale_price'].isna() | (vendas['sale_price'] <= 0)

registrar('precos', 'linhas_apos_deduplicacao_listing_data', len(precos))
registrar('precos', 'diarias_com_multiplas_capturas', int((estatisticas_captura['quantidade_capturas'] > 1).sum()), severidade='atenção')
registrar('precos', 'precos_invalidos_sinalizados', int(precos['flag_preco_invalido'].sum()), 'price', 'erro')
registrar('vendas', 'precos_venda_invalidos_sinalizados', int(vendas['flag_preco_venda_invalido'].sum()), 'sale_price', 'erro')

assert not precos.duplicated(chave_preco).any()
assert vendas['listing_id'].is_unique

resumo_snapshots = pd.DataFrame(
    {
        'base_derivada': ['details', 'hosts', 'mesh', 'precos', 'vendas'],
        'linhas': [len(details), len(hosts), len(mesh), len(precos), len(vendas)],
    }
)
display(resumo_snapshots)

,base_derivada,linhas
0,details,4441
1,hosts,3057
2,mesh,4441
3,precos,59040
4,vendas,8293


In [5]:
def registrar_cobertura(dataset, relacionamento, ids_esquerda, ids_direita):
    esquerda = set(pd.Series(ids_esquerda).dropna().astype('string'))
    direita = set(pd.Series(ids_direita).dropna().astype('string'))
    correspondentes = esquerda & direita
    registrar(dataset, 'ids_esquerda', len(esquerda), relacionamento)
    registrar(dataset, 'ids_correspondentes', len(correspondentes), relacionamento)
    registrar(dataset, 'ids_orfaos_esquerda', len(esquerda - direita), relacionamento, 'atenção')
    registrar(dataset, 'ids_orfaos_direita', len(direita - esquerda), relacionamento, 'atenção')
    registrar(
        dataset,
        'cobertura_percentual',
        round(len(correspondentes) / len(esquerda) * 100, 4) if esquerda else 0,
        relacionamento,
    )


registrar_cobertura('joins', 'details_mesh', details['airbnb_listing_id'], mesh['airbnb_listing_id'])
registrar_cobertura('joins', 'details_hosts', details['owner_id'], hosts['owner_id'])
registrar_cobertura('joins', 'details_precos', details['airbnb_listing_id'], precos['airbnb_listing_id'])

mesh_para_join = mesh.rename(
    columns={
        'latitude': 'mesh_latitude',
        'longitude': 'mesh_longitude',
        'aquisition_date': 'mesh_aquisition_date',
        'quantidade_snapshots': 'mesh_quantidade_snapshots',
    }
)
hosts_para_join = hosts.rename(columns={'quantidade_snapshots': 'host_quantidade_snapshots'})

airbnb = details.merge(
    mesh_para_join,
    on='airbnb_listing_id',
    how='left',
    validate='one_to_one',
    suffixes=('', '_mesh'),
)
linhas_apos_mesh = len(airbnb)
airbnb = airbnb.merge(
    hosts_para_join,
    on='owner_id',
    how='left',
    validate='many_to_one',
    suffixes=('', '_host'),
)
airbnb['tem_localizacao_mesh'] = airbnb['suburb_key'].notna()
airbnb['tem_dados_host'] = airbnb['host_snapshot_date'].notna()
airbnb['tem_precos'] = airbnb['airbnb_listing_id'].isin(precos['airbnb_listing_id'])
airbnb['latitude_analitica'] = airbnb['mesh_latitude']
airbnb['longitude_analitica'] = airbnb['mesh_longitude']
coordenadas_pares = airbnb[['latitude', 'longitude', 'mesh_latitude', 'mesh_longitude']].dropna()
coordenadas_pares = coordenadas_pares.loc[
    coordenadas_pares[['latitude', 'longitude', 'mesh_latitude', 'mesh_longitude']].ne(0).all(axis=1)
]
coordenadas_divergentes = (
    (coordenadas_pares['latitude'] - coordenadas_pares['mesh_latitude']).abs().gt(0.001)
    | (coordenadas_pares['longitude'] - coordenadas_pares['mesh_longitude']).abs().gt(0.001)
)

assert len(airbnb) == len(details) == linhas_apos_mesh
assert airbnb['airbnb_listing_id'].is_unique
registrar('joins', 'linhas_base_airbnb', len(airbnb), 'base_airbnb')
registrar('joins', 'sem_localizacao_mesh', int((~airbnb['tem_localizacao_mesh']).sum()), 'base_airbnb', 'atenção')
registrar('joins', 'sem_dados_host', int((~airbnb['tem_dados_host']).sum()), 'base_airbnb', 'atenção')
registrar('joins', 'sem_precos', int((~airbnb['tem_precos']).sum()), 'base_airbnb', 'atenção')
registrar('joins', 'pares_coordenadas_comparaveis', len(coordenadas_pares), 'base_airbnb')
registrar('joins', 'coordenadas_details_mesh_divergentes', int(coordenadas_divergentes.sum()), 'base_airbnb', 'atenção', 'tolerância absoluta de 0,001 grau')

resumo_joins = pd.DataFrame(registros_qualidade).query("dataset == 'joins'")
display(resumo_joins[['coluna', 'metrica', 'valor', 'severidade']])

,coluna,metrica,valor,severidade
402,details_mesh,ids_esquerda,4441,informativo
403,details_mesh,ids_correspondentes,4441,informativo
404,details_mesh,ids_orfaos_esquerda,0,atenção
405,details_mesh,ids_orfaos_direita,0,atenção
406,details_mesh,cobertura_percentual,100.0,informativo
407,details_hosts,ids_esquerda,3057,informativo
408,details_hosts,ids_correspondentes,3057,informativo
409,details_hosts,ids_orfaos_esquerda,0,atenção
410,details_hosts,ids_orfaos_direita,0,atenção
411,details_hosts,cobertura_percentual,100.0,informativo


In [6]:
def caminho_saida(nome):
    destino = (SAIDAS / nome).resolve()
    assert destino.parent == SAIDAS.resolve(), f'Saída fora do diretório permitido: {destino}'
    return destino


qualidade = pd.DataFrame(registros_qualidade)
qualidade = qualidade.sort_values(['dataset', 'coluna', 'metrica'], kind='stable').reset_index(drop=True)
dicionario_dados = pd.DataFrame(dicionario).sort_values(['dataset', 'coluna']).reset_index(drop=True)

qualidade.to_csv(caminho_saida('qualidade_dados.csv'), index=False, encoding='utf-8')
dicionario_dados.to_csv(caminho_saida('dicionario_dados.csv'), index=False, encoding='utf-8')
airbnb.to_csv(caminho_saida('airbnb_listings.csv'), index=False, encoding='utf-8')
precos.to_csv(caminho_saida('precos_airbnb.csv'), index=False, encoding='utf-8')
vendas.to_csv(caminho_saida('vivareal_listings.csv'), index=False, encoding='utf-8')

linhas_preco_sem_chave = int(precos_raw[chave_preco].isna().any(axis=1).sum())
linhas_details_sem_chave = int(tratados['details']['airbnb_listing_id'].isna().sum())
linhas_vendas_sem_chave = int(tratados['vendas']['listing_id'].isna().sum())
coordenadas_details_zeradas = int((details['latitude'].eq(0) & details['longitude'].eq(0)).sum())

decisoes = f'''# Decisões de preparação dos dados

## Princípios

- Os cinco CSVs de `data/` são somente leitura e tiveram seus hashes verificados antes e depois da auditoria.
- Conversões inválidas viram valores ausentes e são contabilizadas em `qualidade_dados.csv`.
- Campos textuais têm whitespace consecutivo normalizado para um espaço; o conteúdo lexical é preservado.
- Outliers são sinalizados; não são excluídos automaticamente nesta etapa.
- O bairro original é preservado; `suburb_key` remove acentos, espaços e pontuação para expor variações técnicas sem impor equivalências semânticas.
- Preço de diária e preço de venda continuam classificados como valores anunciados, não receita ou transação realizada.

## Chaves e snapshots

- Details: snapshot mais recente por `airbnb_listing_id`; {linhas_details_sem_chave} linha(s) sem a chave ficaram fora da base derivada.
- Hosts: snapshot mais recente por `owner_id`.
- Mesh: snapshot mais recente por `airbnb_listing_id`.
- VivaReal: snapshot mais recente por `listing_id`; {linhas_vendas_sem_chave} linha(s) sem a chave ficaram fora da base derivada.
- Price_AV: captura mais recente por `airbnb_listing_id` e `date`; {linhas_preco_sem_chave} linha(s) sem chave/data ficaram fora da base temporal.
- A quantidade, a primeira e a última captura de cada diária foram preservadas em colunas auxiliares.
- Details possui {coordenadas_details_zeradas} anúncio(s) com latitude/longitude zeradas; as colunas analíticas de coordenadas usam Mesh, que tem cobertura completa após o join.

## Joins

- Details é a base-mãe do Airbnb. Mesh entra por `airbnb_listing_id` com cardinalidade 1:1 após snapshots.
- Hosts entra por `owner_id` com cardinalidade N:1 após snapshots.
- Price_AV permanece em base temporal separada e sua cobertura é registrada.
- VivaReal permanece separado; não existe join direto confiável por identificador com Airbnb.
- Joins são à esquerda para não excluir anúncios silenciosamente; flags indicam disponibilidade das fontes auxiliares.

## Limites para as próximas etapas

- Registros com preços inválidos permanecem nas bases com flags e deverão ser filtrados explicitamente ao calcular métricas.
- Nenhuma hipótese de ocupação foi criada.
- Nenhuma inferência causal ou recomendação de investimento foi realizada.
'''
caminho_saida('decisoes_limpeza.md').write_text(decisoes, encoding='utf-8')

arquivos_saida = sorted(SAIDAS.glob('*'))
inventario_saidas = pd.DataFrame(
    [{'arquivo': arquivo.name, 'bytes': arquivo.stat().st_size} for arquivo in arquivos_saida]
)
display(inventario_saidas)

,arquivo,bytes
0,airbnb_listings.csv,8226356
1,decisoes_limpeza.md,2274
2,dicionario_dados.csv,10172
3,precos_airbnb.csv,6281865
4,qualidade_dados.csv,23124
5,vivareal_listings.csv,4116614


In [7]:
hashes_depois = {caminho.name: sha256(caminho) for caminho in caminhos.values()}
assert hashes_depois == hashes_antes == HASHES_ETAPA_0, 'A pasta data/ foi alterada durante a auditoria.'
assert airbnb['airbnb_listing_id'].is_unique
assert not precos.duplicated(['airbnb_listing_id', 'date']).any()
assert vendas['listing_id'].is_unique
assert set(arquivo.name for arquivo in SAIDAS.iterdir()) == {
    'qualidade_dados.csv',
    'dicionario_dados.csv',
    'airbnb_listings.csv',
    'precos_airbnb.csv',
    'vivareal_listings.csv',
    'decisoes_limpeza.md',
}

display(Markdown(
    f'''## Etapa 1 validada

- **Airbnb:** {len(airbnb):,} anúncios únicos.
- **Preços:** {len(precos):,} combinações únicas de anúncio e data de estadia.
- **VivaReal:** {len(vendas):,} anúncios únicos de venda/locação.
- **Qualidade:** {len(qualidade):,} métricas registradas.
- **Dados brutos:** hashes preservados.
- **Saídas:** restritas a `outputs/auditoria/`.
'''
))

## Etapa 1 validada

- **Airbnb:** 4,441 anúncios únicos.
- **Preços:** 59,040 combinações únicas de anúncio e data de estadia.
- **VivaReal:** 8,293 anúncios únicos de venda/locação.
- **Qualidade:** 423 métricas registradas.
- **Dados brutos:** hashes preservados.
- **Saídas:** restritas a `outputs/auditoria/`.
